### Creating a view to analyse the rankings of the driver over the season based on various metrics
- Sources -> fact_session_results, dim_drivers
- Required columns -> season, driver_id, driver_name, nationality
- Calculated columns -> race_starts (number of races the driver participates), total_points (sum of the points scored by the driver), number_of_wins (if the is_win column is true), number_of_podiums (if the is_podium column is true)

In [0]:
%sql
CREATE OR REPLACE VIEW formula1.gold.v_driver_standings AS
WITH driver_session_summary AS (select 
f.season, d.driver_id, d.driver_name, d.nationality, 
COUNT(*) as race_starts,
SUM(f.points) as total_points,
count_if(f.is_win) as number_of_wins,
count_if(f.is_podium) as number_of_podiums from 
formula1.gold.fact_session_results f join formula1.gold.dim_drivers d on f.driver_id = d.driver_id 
group by f.season, d.driver_id, d.driver_name, d.nationality)
select season, driver_id, driver_name, nationality,rank() over (partition by season order by total_points desc, number_of_wins desc) as standings, race_starts, total_points, number_of_wins, number_of_podiums from driver_session_summary

In [0]:
%sql
select * from formula1.gold.v_driver_standings order by season, standings